# Thai Election OCR Pipeline

Extracts vote counts from 847 scanned Thai election document images (Form สส.6/1) and produces `submission.csv` with 10,053 rows.

In [ ]:
import base64
import json
import os
import re
import threading
import time
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import get_close_matches
from pathlib import Path
from typing import Protocol

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
from typhoon_ocr import ocr_document

load_dotenv()

DATA_DIR = Path("../data")
IMAGES_DIR = DATA_DIR / "images"
CACHE_DIR = Path("../cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OCR_CACHE_CSV = CACHE_DIR / "ocr_cache.csv"
OCR_QUALITY_CACHE_CSV = CACHE_DIR / "ocr_quality_cache.csv"
LLM_CACHE_CSV = CACHE_DIR / "llm_cache.csv"
SAMPLE_LABELS_DIR = DATA_DIR / "sample_labels"

MAX_WORKERS = os.cpu_count() or 4

openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODEL = "google/gemini-2.0-flash-001"
MODEL_CHECK = "google/gemini-2.0-flash-001"

class RateLimiter:
    """Sliding-window rate limiter supporting multiple simultaneous constraints."""

    def __init__(self, limits: list[tuple[int, float]]):
        """
        limits: list of (max_requests, window_seconds) pairs.
        All constraints must be satisfied before a slot is granted.
        """
        self._limits = limits
        self._windows: list[deque] = [deque() for _ in limits]
        self._lock = threading.Lock()

    def acquire(self) -> None:
        """Block until all rate-limit windows have room for one more request."""
        while True:
            with self._lock:
                now = time.monotonic()
                # Evict timestamps outside each window
                for (max_req, window), dq in zip(self._limits, self._windows):
                    while dq and now - dq[0] >= window:
                        dq.popleft()
                # Check if all windows have capacity
                if all(
                    len(dq) < max_req
                    for (max_req, _), dq in zip(self._limits, self._windows)
                ):
                    for dq in self._windows:
                        dq.append(now)
                    return
                # Compute minimum wait: earliest slot that opens across any saturated window
                wait = min(
                    (max_req, window, dq)[2][0] + window - now  # dq[0] + window - now
                    for (max_req, window), dq in zip(self._limits, self._windows)
                    if len(dq) >= max_req
                )
                wait = max(wait, 0.0)
            time.sleep(wait + 0.05)

class OcrBackend(Protocol):
    def __call__(self, image_path: Path) -> str: ...


def _ocr_typhoon(image_path: Path) -> str:
    """Typhoon OCR via typhoon_ocr package."""
    return ocr_document(str(image_path))


_VLM_OCR_PROMPT = (
    "This is a page from a Thai official election result form (สส.6/1). "
    "Your task is to perform a complete, verbatim transcription of every piece of information on this page. "
    "Output only Markdown — no commentary, no explanations, nothing outside the transcription.\n\n"
    "Follow these rules strictly:\n"
    "1. **Transcribe every row** of every table without exception. Never skip, truncate, or summarise rows.\n"
    "2. **Preserve exact Thai text** for all party names (ชื่อพรรค), candidate names (ชื่อ-สกุล), "
    "   section labels, and header fields exactly as printed.\n"
    "3. **Convert Thai numerals** (๐๑๒๓๔๕๖๗๘๙) to Arabic digits (0–9) in all numeric fields.\n"
    "4. **For vote-count tables**, reproduce each data row as a Markdown table row with columns:\n"
    "   | หมายเลข | ชื่อ-สกุล | สังกัดพรรคการเมือง | คะแนน |\n"
    "   Include the header row and a separator row.\n"
    "5. **For summary statistics** (numbered items such as 1.1, 1.2, 2.1, 3.1 …), transcribe each item "
    "   on its own line in the format: `<item number> <label> <value>`.\n"
    "6. **Transcribe all header fields** (province, constituency/district name, polling station, date, "
    "   official signatures section labels) verbatim.\n"
    "7. If a cell is blank or illegible, write `-`.\n"
    "8. Do **not** add any text, commentary, or formatting that is not present in the image."
)


def _ocr_openrouter_vlm(image_path: Path, model: str = "google/gemini-2.0-flash-001") -> str:
    """VLM-based OCR via OpenRouter: encode image as base64, ask the model to transcribe."""
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = openrouter.chat.completions.create(
        model=model,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64}"},
                },
                {
                    "type": "text",
                    "text": _VLM_OCR_PROMPT,
                },
            ],
        }],
        temperature=0,
    )
    return resp.choices[0].message.content or ""


# ---------------------------------------------------------------------------
# OCR backend registry — add new backends here
# ---------------------------------------------------------------------------
OCR_BACKENDS: dict[str, OcrBackend] = {
    "typhoon": _ocr_typhoon,
    "gemini-2.5-flash": lambda p: _ocr_openrouter_vlm(p, model="google/gemini-2.5-flash"),
    # "gpt-4o": lambda p: _ocr_openrouter_vlm(p, model="openai/gpt-4o"),  # example future backend
}

# ── Select active backend and matching rate limiter ─────────────────────────
# Change OCR_BACKEND_NAME to switch providers.
OCR_BACKEND_NAME = "gemini-2.5-flash"   # "typhoon" | "gemini-2.5-flash" | ...
OCR_RATE_LIMITS: dict[str, list[tuple[int, float]]] = {
    "typhoon":         [(2, 1.0), (20, 60.0)],
    "gemini-2.5-flash": [(10, 1.0), (60, 60.0)],
}

ocr_backend: OcrBackend = OCR_BACKENDS[OCR_BACKEND_NAME]
ocr_rate_limiter = RateLimiter(limits=OCR_RATE_LIMITS.get(OCR_BACKEND_NAME, [(5, 1.0)]))

print(f"Setup complete. OCR backend: {OCR_BACKEND_NAME!r}")


Setup complete. OCR backend: 'gemini-2.5-flash'


In [2]:
# Cell 2 — Group Pages by doc_id

def get_page_order(name: str) -> int:
    """constituency_10_1.png → 1, constituency_10_1_page2.png → 2"""
    m = re.search(r"_page(\d+)\.png$", name)
    return int(m.group(1)) if m else 1


def get_doc_id(name: str) -> str:
    """Strip _pageN.png suffix to get doc_id."""
    return re.sub(r"_page\d+\.png$", "", name).removesuffix(".png")


doc_pages: dict[str, list[Path]] = defaultdict(list)
for img in IMAGES_DIR.glob("*.png"):
    doc_pages[get_doc_id(img.name)].append(img)

for doc_id in doc_pages:
    doc_pages[doc_id].sort(key=lambda p: get_page_order(p.name))

template = pd.read_csv(DATA_DIR / "submission_template.csv", dtype=str)
template = template.dropna(subset=["doc_id", "party_name"])
template = template[template["party_name"].str.strip() != ""]

doc_parties: dict[str, list[str]] = (
    template.groupby("doc_id")["party_name"].apply(list).to_dict()
)

print(f"Documents : {len(doc_pages)}")
print(f"Pages     : {sum(len(v) for v in doc_pages.values())}")
print(f"Template  : {len(template)} rows across {len(doc_parties)} docs")

Documents : 300
Pages     : 847
Template  : 10039 rows across 300 docs


In [ ]:
# Cell 3 — Batch OCR  →  ocr_cache.csv
# Cache schema: id (int, doc order), doc_id (str), page (int, 1-based), ocr_text
# Sorted by id ASC, page ASC.
# Concurrency: os.cpu_count() workers; RateLimiter enforces backend-specific limits.
# Blank ocr_text rows (from prior rate-limit failures) are detected and re-queued.

_ocr_write_lock = threading.Lock()

_doc_id_list: list[str] = sorted(doc_pages.keys())
_doc_id_to_idx: dict[str, int] = {d: i + 1 for i, d in enumerate(_doc_id_list)}


def _load_ocr_cache() -> dict[tuple[str, int], str]:
    """Returns {(doc_id, page): ocr_text}. Blank entries kept for retry detection."""
    if OCR_CACHE_CSV.exists():
        df = pd.read_csv(OCR_CACHE_CSV, dtype={"id": int, "page": int, "doc_id": str, "ocr_text": str})
        df["ocr_text"] = df["ocr_text"].fillna("")
        return {(row.doc_id, int(row.page)): row.ocr_text for row in df.itertuples()}
    return {}


def _save_ocr_cache(cache: dict[tuple[str, int], str]) -> None:
    rows = [
        {"id": _doc_id_to_idx[doc_id], "doc_id": doc_id, "page": page, "ocr_text": text}
        for (doc_id, page), text in cache.items()
    ]
    df = pd.DataFrame(rows, columns=["id", "doc_id", "page", "ocr_text"])
    df.sort_values(["id", "page"]).reset_index(drop=True).to_csv(OCR_CACHE_CSV, index=False)


def _ocr_one_page(doc_id: str, page_num: int, page: Path) -> tuple[str, int, str | None]:
    """
    Returns (doc_id, page_num, ocr_text).
    ocr_text is None if already cached with non-blank text (skip signal).
    Uses the globally selected ocr_backend and ocr_rate_limiter.
    """
    with _ocr_write_lock:
        cached = ocr_cache.get((doc_id, page_num))
        if cached is not None and cached.strip():
            return doc_id, page_num, None  # already good

    ocr_rate_limiter.acquire()
    for attempt in range(3):
        try:
            text = ocr_backend(page)
            if text and text.strip():
                return doc_id, page_num, text
            print(f"  OCR returned blank [{doc_id} p{page_num}] attempt {attempt + 1} (backend={OCR_BACKEND_NAME!r})")
            time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  OCR attempt {attempt + 1} failed [{doc_id} p{page_num}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, page_num, ""  # persist blank so row exists; re-queued on next run


ocr_cache: dict[tuple[str, int], str] = _load_ocr_cache()

blank_keys = {k for k, v in ocr_cache.items() if not v.strip()}
good_keys  = {k for k, v in ocr_cache.items() if v.strip()}
print(f"OCR cache: {len(good_keys)} good, {len(blank_keys)} blank (will retry)")
print(f"Active backend: {OCR_BACKEND_NAME!r}")

work: list[tuple[str, int, Path]] = []
for doc_id, pages in doc_pages.items():
    for page_num, page in enumerate(pages, start=1):
        key = (doc_id, page_num)
        if key not in ocr_cache or not ocr_cache[key].strip():
            work.append((doc_id, page_num, page))

print(f"Pages to OCR (new + retry): {len(work)}")

MAX_WORKERS = os.cpu_count() or 4
print(f"Workers: {MAX_WORKERS}  (rate-limited by RateLimiter)")

if work:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_ocr_one_page, doc_id, page_num, page): (doc_id, page_num)
            for doc_id, page_num, page in work
        }
        with tqdm(total=len(work), desc=f"Batch OCR [{OCR_BACKEND_NAME}]") as pbar:
            for future in as_completed(futures):
                doc_id, page_num, text = future.result()
                if text is not None:
                    with _ocr_write_lock:
                        ocr_cache[(doc_id, page_num)] = text
                        _save_ocr_cache(ocr_cache)
                pbar.update(1)

still_blank = [(d, p) for (d, p), v in ocr_cache.items() if not v.strip()]
if still_blank:
    print(f"WARNING: {len(still_blank)} pages still blank after retries: {still_blank[:10]}")
else:
    print("All pages have non-blank OCR text.")

doc_ocr: dict[str, str] = {
    doc_id: "\n\n---PAGE BREAK---\n\n".join(
        ocr_cache.get((doc_id, page_num), "")
        for page_num, _ in enumerate(pages, start=1)
    )
    for doc_id, pages in doc_pages.items()
}

print(f"OCR complete. ocr_cache.csv → {len(ocr_cache)} pages total.")

OCR cache: 844 good, 0 blank (will retry)
Active backend: 'gemini-2.5-flash'
Pages to OCR (new + retry): 3
Workers: 16  (rate-limited by RateLimiter)


Batch OCR [gemini-2.5-flash]:   0%|          | 0/3 [00:00<?, ?it/s]

All pages have non-blank OCR text.
OCR complete. ocr_cache.csv → 847 pages total.


In [6]:
# Cell 4 — Batch OCR Quality Check  →  ocr_quality_cache.csv
# Schema: id (int), doc_id, passed (bool), issues (str)
# Sorted by id ASC. Concurrent — one LLM call per doc.

OCR_QUALITY_PROMPT = """You are a quality checker for OCR output from scanned Thai election forms (สส.6/1).

Inspect the OCR text below and answer:
1. Are Thai party names legible (not garbled)?
2. Are vote count columns present and contain numeric values?
3. Is the table structure reasonably intact across all pages?

Return ONLY valid JSON with exactly these keys:
{{
  "passed": true or false,
  "issues": "short description of problems, or empty string if passed"
}}

OCR text:
{ocr_text}"""

_ocr_quality_lock = threading.Lock()


def _load_ocr_quality_cache() -> dict[str, dict]:
    if OCR_QUALITY_CACHE_CSV.exists():
        df = pd.read_csv(OCR_QUALITY_CACHE_CSV, dtype={"id": int, "doc_id": str, "passed": str, "issues": str}).fillna("")
        return {row.doc_id: {"passed": row.passed == "True", "issues": row.issues} for row in df.itertuples()}
    return {}


def _save_ocr_quality_cache(cache: dict[str, dict]) -> None:
    rows = [
        {"id": _doc_id_to_idx[doc_id], "doc_id": doc_id, "passed": v["passed"], "issues": v["issues"]}
        for doc_id, v in cache.items()
    ]
    pd.DataFrame(rows, columns=["id", "doc_id", "passed", "issues"]) \
      .sort_values("id").reset_index(drop=True) \
      .to_csv(OCR_QUALITY_CACHE_CSV, index=False)


def _check_ocr_quality_one(doc_id: str, ocr_text: str) -> tuple[str, dict | None]:
    """Returns (doc_id, result). result is None if already cached."""
    with _ocr_quality_lock:
        if doc_id in ocr_quality_cache:
            return doc_id, None
    prompt = OCR_QUALITY_PROMPT.format(ocr_text=ocr_text[:8000])
    for attempt in range(3):
        try:
            resp = openrouter.chat.completions.create(
                model=MODEL_CHECK,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = json.loads(resp.choices[0].message.content)
            return doc_id, {"passed": bool(raw.get("passed", False)), "issues": str(raw.get("issues", ""))}
        except Exception as e:
            print(f"  OCR quality attempt {attempt+1} failed [{doc_id}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, {"passed": False, "issues": "quality check API error"}


ocr_quality_cache: dict[str, dict] = _load_ocr_quality_cache()
work_q4 = [(doc_id, ocr_text) for doc_id, ocr_text in doc_ocr.items() if doc_id not in ocr_quality_cache]
print(f"OCR quality cache: {len(ocr_quality_cache)} done, {len(work_q4)} to check")

if work_q4:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_check_ocr_quality_one, doc_id, ocr_text): doc_id for doc_id, ocr_text in work_q4}
        with tqdm(total=len(work_q4), desc="Batch OCR quality check") as pbar:
            for future in as_completed(futures):
                doc_id, result = future.result()
                if result is not None:
                    with _ocr_quality_lock:
                        ocr_quality_cache[doc_id] = result
                        _save_ocr_quality_cache(ocr_quality_cache)
                pbar.update(1)

failed_ocr = [d for d, v in ocr_quality_cache.items() if not v["passed"]]
print(f"\nOCR quality: {len(ocr_quality_cache) - len(failed_ocr)}/{len(ocr_quality_cache)} passed")
if failed_ocr:
    print(f"Failed docs ({len(failed_ocr)}): {failed_ocr[:10]}")

OCR quality cache: 0 done, 300 to check


Batch OCR quality check:   0%|          | 0/300 [00:00<?, ?it/s]


OCR quality: 164/300 passed
Failed docs (136): ['constituency_10_20', 'constituency_10_1', 'constituency_10_2', 'constituency_10_11', 'constituency_10_23', 'constituency_10_12', 'constituency_10_18', 'constituency_10_17', 'constituency_10_14', 'constituency_10_22']


In [4]:
# Cell 5 — Batch LLM Vote Extraction  →  llm_cache.csv
# Schema: id (int), doc_id, party_name, votes
# Sorted by id ASC, party_name ASC. Concurrent — one LLM call per doc.

CONSTITUENCY_PROMPT = """You are extracting vote counts from a scanned Thai election result form (สส.6/1).
This is a constituency (แบบแบ่งเขต) document — each row has a candidate name, party, and vote count.

For each party in the list, find and return its total vote count.
Return ONLY valid JSON: {{"party_name": vote_count, ...}}
All vote counts must be integers (Arabic digits 0-9). Use 0 if a party is not found.

Parties to extract:
{party_list}

OCR text:
{ocr_text}"""

PARTY_LIST_PROMPT = """You are extracting vote counts from a scanned Thai election result form (สส.6/1).
This is a party-list (แบบบัญชีรายชื่อ) document — 57 parties numbered 1-57, no candidate names.

For each party in the list, find and return its vote count.
Return ONLY valid JSON: {{"party_name": vote_count, ...}}
All vote counts must be integers (Arabic digits 0-9). Use 0 if a party is not found.

Parties to extract:
{party_list}

OCR text:
{ocr_text}"""

_llm_lock = threading.Lock()


def _load_llm_cache() -> dict[str, dict[str, int]]:
    if not LLM_CACHE_CSV.exists():
        return {}
    df = pd.read_csv(LLM_CACHE_CSV, dtype=str)  # read all as str first to avoid NaN coercion
    df = df.dropna(subset=["doc_id", "party_name", "votes"])  # drop any incomplete rows
    df = df[df["party_name"].str.strip() != ""]               # drop blank party names
    result: dict[str, dict[str, int]] = defaultdict(dict)
    for row in df.itertuples():
        try:
            result[str(row.doc_id)][str(row.party_name)] = int(float(row.votes))
        except (ValueError, TypeError):
            pass
    return dict(result)


def _save_llm_cache(cache: dict[str, dict[str, int]]) -> None:
    rows = [
        {"id": _doc_id_to_idx[doc_id], "doc_id": doc_id, "party_name": party, "votes": votes}
        for doc_id, parties in cache.items()
        for party, votes in parties.items()
    ]
    pd.DataFrame(rows, columns=["id", "doc_id", "party_name", "votes"]) \
      .sort_values(["id", "party_name"]).reset_index(drop=True) \
      .to_csv(LLM_CACHE_CSV, index=False)


def match_to_template(extracted: dict[str, int], parties: list[str], doc_id: str = "") -> dict[str, int]:
    result = {p: 0 for p in parties}
    unmatched = []
    for llm_name, votes in extracted.items():
        # Guard: skip non-string or blank keys (e.g. NaN leaked from CSV)
        if not isinstance(llm_name, str) or not llm_name.strip():
            continue
        if llm_name in result:
            result[llm_name] = votes
        else:
            close = get_close_matches(llm_name, parties, n=1, cutoff=0.7)
            if close:
                result[close[0]] = votes
            else:
                unmatched.append(llm_name)
    if unmatched:
        print(f"  WARNING [{doc_id}]: unmatched: {unmatched}")
    return result


def _extract_votes_one(doc_id: str, ocr_text: str, parties: list[str]) -> tuple[str, dict[str, int] | None]:
    """Returns (doc_id, result). result is None if already cached."""
    with _llm_lock:
        if doc_id in llm_cache:
            return doc_id, None
    is_party_list = doc_id.startswith("party_list")
    prompt = (PARTY_LIST_PROMPT if is_party_list else CONSTITUENCY_PROMPT).format(
        party_list="\n".join(f"- {p}" for p in parties),
        ocr_text=ocr_text[:12000],
    )
    for attempt in range(3):
        try:
            resp = openrouter.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0,
            )
            raw = json.loads(resp.choices[0].message.content)
            # Coerce all keys to str and values to int; skip unparseable entries
            result: dict[str, int] = {}
            for k, v in raw.items():
                try:
                    result[str(k)] = int(float(v))
                except (TypeError, ValueError):
                    pass
            return doc_id, result
        except Exception as e:
            print(f"  LLM attempt {attempt+1} failed [{doc_id}]: {e}")
            time.sleep(2 ** attempt)
    return doc_id, {p: 0 for p in parties}


llm_cache: dict[str, dict[str, int]] = _load_llm_cache()
work_q5 = [
    (doc_id, doc_ocr[doc_id], doc_parties[doc_id])
    for doc_id in doc_ocr
    if doc_id in doc_parties and doc_id not in llm_cache
]
print(f"LLM cache: {len(llm_cache)} done, {len(work_q5)} to extract")

if work_q5:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(_extract_votes_one, doc_id, ocr_text, parties): doc_id
            for doc_id, ocr_text, parties in work_q5
        }
        with tqdm(total=len(work_q5), desc="Batch LLM extraction") as pbar:
            for future in as_completed(futures):
                doc_id, result = future.result()
                if result is not None:
                    with _llm_lock:
                        llm_cache[doc_id] = result
                        _save_llm_cache(llm_cache)
                pbar.update(1)

print(f"LLM extraction complete. llm_cache.csv has {len(llm_cache)} docs.")

LLM cache: 0 done, 300 to extract


Batch LLM extraction:   0%|          | 0/300 [00:00<?, ?it/s]

  LLM attempt 1 failed [constituency_10_26]: Expecting property name enclosed in double quotes: line 2 column 15 (char 16)
  LLM attempt 2 failed [constituency_10_26]: Expecting property name enclosed in double quotes: line 2 column 15 (char 16)
  LLM attempt 3 failed [constituency_10_26]: Expecting property name enclosed in double quotes: line 2 column 15 (char 16)
LLM extraction complete. llm_cache.csv has 300 docs.


In [5]:
# Cell 7 — Assemble Final Results
# Priority per doc:
#   1. match_to_template(llm_cache) — LLM extraction matched to template
#   2. {party: 0}                   — fallback if nothing available

all_results: dict[str, dict[str, int]] = {}

for doc_id in doc_pages:
    parties = doc_parties.get(doc_id, [])
    if not parties:
        continue

    if doc_id in llm_cache:
        all_results[doc_id] = match_to_template(llm_cache[doc_id], parties, doc_id)
    else:
        all_results[doc_id] = {p: 0 for p in parties}

zero_docs = [d for d, v in all_results.items() if all(votes == 0 for votes in v.values())]
print(f"Assembled {len(all_results)} docs  |  all-zero docs: {len(zero_docs)}")

Assembled 300 docs  |  all-zero docs: 1


In [8]:
# Cell 8 — Validate Against Sample Labels

def validate(all_results: dict[str, dict[str, int]]) -> None:
    correct, total = 0, 0
    for label_file in sorted(SAMPLE_LABELS_DIR.glob("*.json")):
        doc_id = label_file.stem
        label = json.loads(label_file.read_text(encoding="utf-8"))
        predicted = all_results.get(doc_id)

        if predicted is None:
            print(f"  MISSING [{doc_id}]: not in all_results — did earlier cells run?")
            continue

        doc_correct, doc_total = 0, 0
        for entry in label["results"]:
            party, expected = entry["party"], entry["votes"]
            got = predicted.get(party, 0)
            if expected == got:
                doc_correct += 1
            else:
                print(f"  MISMATCH [{doc_id}] {party}: expected={expected}, got={got}")
            doc_total += 1

        correct += doc_correct
        total += doc_total

        # ocr_q  = ocr_quality_cache.get(doc_id, {})
        # source = "llm" if doc_id in llm_cache else "fallback"
        # print(
        #     f"  {doc_id}: {doc_correct}/{doc_total}"
        #     f"  src={source}"
        #     f"  ocr_pass={ocr_q.get('passed', '?')}"
        # )

    if total:
        print(f"\nValidation: {correct}/{total} exact matches ({100 * correct / total:.1f}%)")
    else:
        print("No sample labels found.")

validate(all_results)


Validation: 197/197 exact matches (100.0%)


In [13]:
# Cell 9 — Write submission.csv
# Load template as the base; fill votes column where doc_id matches all_results.

sample_submission = pd.read_csv(DATA_DIR / "submission_template.csv", dtype=str)
sample_submission["votes"] = sample_submission.apply(
    lambda row: all_results.get(row["doc_id"], {}).get(row["party_name"], 0),
    axis=1,
).astype(int)

sample_submission[["id", "votes"]].to_csv("../submission.csv", index=False)

total_rows   = len(sample_submission)
nonzero_rows = int((sample_submission["votes"] > 0).sum())
zero_rows    = total_rows - nonzero_rows
print(f"Saved submission.csv: {total_rows} rows")
print(f"  non-zero: {nonzero_rows}  |  zero: {zero_rows}")
if zero_rows == total_rows:
    print("WARNING: all votes are 0 — check that cells 3-7 completed successfully.")
print(sample_submission[["id", "votes"]].head(10))


Saved submission.csv: 10053 rows
  non-zero: 10002  |  zero: 51
                     id  votes
0   constituency_10_1_1  14813
1   constituency_10_1_2  14368
2   constituency_10_1_3    979
3   constituency_10_1_4    244
4   constituency_10_1_5    351
5   constituency_10_1_6  34167
6   constituency_10_1_7   6030
7   constituency_10_1_8   1023
8   constituency_10_1_9   2075
9  constituency_10_1_10    168
